In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 5050 Laptop GPU


In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0.dev20260126+cu128
CUDA available: True


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models

In [3]:
device = torch.device("cpu")
if (torch.cuda.is_available()):
    device = torch.device("cuda")
elif torch.backends.mps.is_built() and torch.backends.mps.is_available():
    device = torch.device("mps")

print(device)

cuda


In [5]:
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

train_set = torchvision.datasets.ImageFolder(root='../pathogen/train', transform=transform_train)
test_set = torchvision.datasets.ImageFolder(root='../pathogen/test', transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=32, shuffle = True, pin_memory=True, num_workers=4)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=32, shuffle = False, pin_memory=True, num_workers=4)

print(f"Classes: {train_set.classes}")

Classes: ['bacteria', 'fungus', 'healthy', 'pests', 'virus']


In [6]:
model = models.resnet18(weights='DEFAULT')

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
print("Training started")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        if i % 100 == 99:
            print(f'[{epoch + 1}/{epochs}, {i + 1:5d}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0
    
    ## VALIDATION ##
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f'--> Epoch {epoch + 1} finished. Test Loss: {test_loss / len(test_loader):.3f} | Accuracy: {accuracy:.2f}%')
    
print('Finished Training')

Training started
[1/10,   100] loss: 0.972
[1/10,   200] loss: 0.703
[1/10,   300] loss: 0.638
[1/10,   400] loss: 0.622
[1/10,   500] loss: 0.583
[1/10,   600] loss: 0.508
[1/10,   700] loss: 0.515
[1/10,   800] loss: 0.469
[1/10,   900] loss: 0.495
[1/10,  1000] loss: 0.473
[1/10,  1100] loss: 0.479
[1/10,  1200] loss: 0.483
[1/10,  1300] loss: 0.451
[1/10,  1400] loss: 0.444
[1/10,  1500] loss: 0.402
[1/10,  1600] loss: 0.452
[1/10,  1700] loss: 0.425
[1/10,  1800] loss: 0.389
[1/10,  1900] loss: 0.404
[1/10,  2000] loss: 0.404
[1/10,  2100] loss: 0.390
[1/10,  2200] loss: 0.390
[1/10,  2300] loss: 0.397
[1/10,  2400] loss: 0.369
[1/10,  2500] loss: 0.388
[1/10,  2600] loss: 0.389
[1/10,  2700] loss: 0.361
[1/10,  2800] loss: 0.387
[1/10,  2900] loss: 0.351
[1/10,  3000] loss: 0.383
[1/10,  3100] loss: 0.357
[1/10,  3200] loss: 0.335
[1/10,  3300] loss: 0.334
[1/10,  3400] loss: 0.353
[1/10,  3500] loss: 0.357
[1/10,  3600] loss: 0.329
[1/10,  3700] loss: 0.347
[1/10,  3800] loss: 0

In [ ]:
MODEL_PATH = "plant_model_pytorch.pth"

torch.save(model.state_dict(), MODEL_PATH)

print(f"PyTorch model saved as {MODEL_PATH}")



In [ ]:
import json

class_names = train_set.classes

with open('class_names.json', 'w') as f:
    json.dump(class_names, f)